**Encoder-Only Transformers for Natural Language Understanding:**
- Release of BERT in 2018 proved that an encoder-only transformer can tackle a wide variety of natural language processing tasks: sentence classification, token classification, multiple choice question answering, etc. BERT also confirmed the effectiveness of self-supervised pretraining on a large corpus for transfer learning: BERT can achieve excellent performance on many tasks, just by fine-tuning on a fairly small dataset for each task. 
- Encoder-only models are generally not used for text generation tasks, such as autocompletion, translation, summarisation, or chatbots, because they are much slower at this than decoders. Decoders are faster because they are causal, so a good implementation can cache and reuse its previous state when predicting a new token. Conversely, encoders use nonmasked multi-head attention layers only, so they are naturally bidirectional:

**BERT's Architecture:**
- Almost identical to the original transformer's encoder, with just 3 differences
  - It is much bigger. BERT-base has 12 encoder blocks, 12 attention heads, and 768-dimensional embeddings, and BERT-large has 24 blocks, 16 heads and 1024 dimensions. Also uses trainable positional embeddings and supports input sentences up to 512 tokens
  - It applies layer-norm just before each sublayer (attention or feedforward) rather than after each skip connection. pre-LN as opposed to post-LN, and it ensures that the inputs of each sublayer are normalized which stabilizes training and reduces sensitivity to weight initialization. 
  - It lets you input sentence in two segments if needed. Useful for tasks that require a pair of input sentences, such as natual language inference or mcqs. To pass 2 sentences to BERT - must first append a separation token [SEP] to each one, then concatenate them. Furthermore, a trainable segment embedding is added to each token's representation: segment embedding #0 is added to all tokens within segment #0, and segment embedding #1 is added to all tokens within segment #1. Positional encodings are also added to each token's representation, as usual (relative to the full input sequence, not relative to the individual segments)

**BERT Pretraining:**
- Authors proposed 2 self-supervised pretraining tasks:
  - Masked language model (MLM)
    - Each token in a sentence has a 15% probablility of being replaced with a mask token, and the model is trained to predict what the original tokens were. For example, if the original sentence is "She had fun at the birthday party", then the model may be given the sentence "She [MASK] fun at the [MASK] party", and it must predict the original sentence: the loss is only computed on the mask token outputs
    - To be more precise, some of the masked tokens are not truly masked: 10% are instead are instead replaced by random tokens, and 10% left alone. Random tokens force the model to perform well even when mask tokens are absent
  - Next Sentence Prediction (NSP)
    - Model is trained to predict whether 2 sentences are consecutive or not. Binary classification task which the authors chose to implement by introducing a new class token [CLS]: this token is inserted at the beginning of the input sentence (position #0, segment #0) and during training the encoder's output, this token is passed through a binary classification head (i.e. a linear layer with a single unit followed by a sigmoid function), and trained using nn.BCELoss, or just a linear layer with a single unit trained using nn.BCEWithLogitsLoss

- BERT was pretrained on both MLM and NSP simultaneously, using a large corpus of text. Goal of NSP was to make the class token's contextualized embedding a good representation of the whole input sequence/ At first, it seemed that it indeed produced good sentence embeddings, but it was later shown that simply pooling all the contextualised embeddings (e.g. by computing their mean) yielded better results. (NSP was later proven not to help much at all)


- You may want to train a BERT model from scratch, for example if dealing with a domain-specific corpus of text. One option would be building BERT yourself by writing the entire transformer architecture out from scratch then preprocess the dataset according to the MLM algorithm and train the model.
- However, there is an easier way, using the Transformers library.  

Lets start by creating a tokenizer and a randomly initialized BERT model. For simplicity, we use a pretrained tokenizer (but of course can and should train one from scratch instead if preferred). Also make sure to tweak the BertConfig depending on your training budget and the size and complexity of your dataset...

In [1]:
from transformers import BertConfig, BertForMaskedLM, BertTokenizerFast
# config class (BertConfig) - this is the config class to store the configuration of a BertModel. Its used to instantiate a BERT model according to the specified
# arguments, defining the model architecture.
# BertForMaskedLM - BERT model with a language modeling head on top. model inherits from pretrained model. 
# BertTokenizerFast -> much faster because its implemented in rust
# BertModel v. BertForMaskedLM => Base BertModel: just the encoder stack (embeddings + Transformer layers) no task specific head
# BertForMaskedLM : bert + MLM head. MLM head projects token embeddings -> vocab size and predicts masked tokens -> logits: [B, seq_len, vocab_size]

In [2]:
bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [3]:
bert_tokenizer.vocab_size

30522

In [4]:
# setting the model config
# instantiates a bert model with this config
config = BertConfig(
    vocab_size = bert_tokenizer.vocab_size, #bert_tokenizer.vocab_size : halving this because of my training budget
    hidden_size = 128, # the dimensionality of the encoder layers
    num_hidden_layers = 2, # hidden layers in the transformer encoder
    num_attention_heads=4, # number of attention heads for each attention layer in the encoder
    intermediate_size=512, # dimensionality of the intermediate or feedforward layer
    max_position_embeddings = 128 # the maximum sequence length this model will ever be used with - used to set the width of the learnable positional encodings matrix
)

In [6]:
bert = BertForMaskedLM(config) # random weights init

Now, lets download the wikitext dataset and tokenize it...

In [8]:
from datasets import load_dataset, load_dataset_builder

In [10]:
mlm_dataset = load_dataset("wikitext","wikitext-2-raw-v1",split="train")

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [11]:
mlm_dataset

Dataset({
    features: ['text'],
    num_rows: 36718
})

Exploring the dataset

In [16]:
for i in range(10):
    print(mlm_dataset[i]['text'])


 = Valkyria Chronicles III = 


 Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . 

 The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series 

In [17]:
def tokenize(example, tokenizer=bert_tokenizer):
    return tokenizer(example["text"], truncation=True, max_length=128, padding="max_length") # padding=max_length means pad every sequence to exactly the max_length

In [18]:
mlm_dataset = mlm_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

this is where mlm now comes in - we create a data collator, whose role is to bundle samples into batches, and we set its mlm argument to True to activate MLM, and also set mlm_probability to 0.15: each token has a 15% probability of being masked (or possibly randomized or left alone). we also pass the tokenizer to the data collator: it lets the data collator know the masking and padding token ids as well as the vocabulary size (which is needed to sample random token ids). with that we need to specify the TrainingArguments, pass everything to the Trainer and call the train method

In [22]:
mlm_dataset[0].keys()

dict_keys(['text', 'input_ids', 'token_type_ids', 'attention_mask'])

In [20]:
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling

In [33]:
bert_tokenizer.convert_ids_to_tokens(200)

'[unused195]'

In [35]:
args = TrainingArguments(
    output_dir="./my_bert",
    num_train_epochs=5,
    per_device_train_batch_size=16
)

mlm_collator = DataCollatorForLanguageModeling(bert_tokenizer, mlm=True, mlm_probability=0.15)

trainer = Trainer(
    model = bert,
    args = args,
    train_dataset = mlm_dataset,
    data_collator = mlm_collator
)

In [36]:
trainer_output = trainer.train()

/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
500,8.877200
1000,7.488500
1500,7.299900
2000,7.216100
2500,7.171200
3000,7.135500
3500,7.131900
4000,7.070700
4500,7.079500
5000,7.018400


/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' 

once the model is pretrained, you can try it out using the pipelines API:

In [39]:
from transformers import pipeline
import torch

In [40]:
torch.manual_seed(42)

In [41]:
fill_mask = pipeline("fill-mask", model=bert, tokenizer=bert_tokenizer)

Device set to use mps:0


In [42]:
top_predictions = fill_mask("The capital of [MASK] is Rome.")

In [46]:
top_predictions[0]

{'score': 0.056581996381282806,
 'token': 1010,
 'token_str': ',',
 'sequence': 'the capital of, is rome.'}

In [47]:
top_preds2 = fill_mask("The [MASK] really liked the game")

In [48]:
top_preds2[0]

{'score': 0.12272864580154419,
 'token': 1996,
 'token_str': 'the',
 'sequence': 'the the really liked the game'}

Model is a bit terrible. because we trained for a few epochs. To get better training results we'd need to train for a very long time. Thats why most people opt to fintune

**BERT Fine-tuning:**
- for sentence classification tasks such as sentiment analysis, all output tokens are ignored except the for the first one which corresponds to the class token, and a new classification head replaces the NSP binary classification head. You can then finetine the whole model using cross-entropy loss, optionally setting a lower learning rate for the lower layers, or freezing BERT altogether during the first few epochs. Using the same approach can tackle other sentence classification tasks, for example 
- for token classification, the classification head is applied to every token. for example bert can be finetuned for named entity recognition (NER), where the model tags part of the text that corresponds to names, dates, places, organisations and other entities....Same application can be done for other token classification tasks, such as tagging grammatical errors, analyzing sentiment at the token level, locating subjects, nouns and verbs
- <img src="/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/resource_images/Screenshot 2025-12-29 at 11.20.52.png">
- BERT can also be used to classify pairs of sentences. it works exactly like sentence classification - except u pass 2 sentences instead of one. (remember BERT has segment encodings added to the sentence embeddings). this can be useful for natural language inference (NLI) where the model must determine whether sentence A entails sentence B, or contradicts it, or neither.
- For MCQ question answering BERT is caalled once for each possible answer, placing the question in segment #0 and the possible answer in segment #1. for each answer the class token output is passed through a linear layer with a single unit producing a score. Once we have all the answer scores, we can convert them to probabilities using a softmax layer and can use the cross entropy loss for finetuning.
- <img src="/Users/blaise/Documents/ML/Machine-Learning-and-Big-Data-Analytics/resource_images/Screenshot 2025-12-29 at 11.28.18.png">
- BERT is also great for extractive question answering: you ask it a question (#segment 0) about some text called the context in segment #1, and BERT must locate the answer within the context. For this, you can add a linear layer with 2 units on top of bert to output 2 scores per tokena start score and an end score. During finetuning, you can treat them as logits for two separate binary classification tasks: the first determines whether a token is the first token in the answer and the second determines whether it is the last. At inference time, we select the pair of indices i and j that maximizes the sum of the start score of token i and the end score of token j, subject to i<=j and j-i+1 <= maximum_acceptable answer length
- BERT authors also showed that BERT could be finetuned to measure semantic textual similarity (STS)- 2 sentences and it outputs a score that indicates how semantically similar the sentences are. So, if u need to find the most similar pair of sentences, in a dataset with N ssentences - u need to run bert on o(N2) pairs: could take hours if the dataset is large. Instead, its preferable to use a model such as sentence-BERT (SBERT) a variant of BERT that was finetuned to produce good sentence embeddings. run each sentence through sbert to get its sentence embedding, then measure the similarity between each pair of sentence embeddings using a similarity measure such as the cosine similarity (e.g. pytorch's f.cosine_similarity() function). This is the cosine of the angle between 2 vectors, so its value ranges from -1 (complete opposite) to +1 (perfectly aligned). Since measuring the cosine similarity is much faster than running BERT, and since the model processes much shorter inputs the process is faster. 
 - Sentence embedding can also be extremely useful in many other applications:
   - Text clustering to organize and better understand your data
     - You can process a large number of documents through SBERT to obtain their sentence embeddings, then apply a clustering algorithm such as k-means on the embeddngs to group the documents based on a semantic similarity. Often helps to reduce the dimensionality before running the clustering algo e.g. using PCA or UMAP
   - Semantic Search: 
     - find documents based on the query's meaning - rather than just keyword matching. First, encode your documents (or chunks of documents) using SBERT and store the sentence embeddings. When a user submits a search query, encode it using SBERT and find the documents whose embeddings are most similar to the query's embedding e.g. based on cosine similarity
   - Reranking search results:
     - have an existing search system you dont want to replace, improve it by reranking the search results based on the semantic similarity with the query.

Many variants of SBERT - easiest way to download them is with the sentence transformers library. the following code downloads the all-MiniLM-L6-v3 model, fast and lightweight but still produces high-quality sentence embeddings.

In [50]:
from sentence_transformers import SentenceTransformer

In [51]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [52]:
sentences = ["She's shopping", "She bought some shoes","She's working"]

In [53]:
embeddings = model.encode(sentences, convert_to_tensor=True)

In [54]:
embeddings

tensor([[-0.0051,  0.0148,  0.0195,  ..., -0.0714,  0.0087, -0.0277],
        [-0.0629,  0.0060,  0.0634,  ..., -0.0852, -0.0408, -0.0386],
        [-0.0751, -0.0433,  0.0261,  ...,  0.0121,  0.0883,  0.0188]],
       device='mps:0')

In [55]:
embeddings.shape

torch.Size([3, 384])

In [56]:
similarities = model.similarity(embeddings, embeddings)

In [57]:
similarities

tensor([[1.0000, 0.6328, 0.5841],
        [0.6328, 1.0000, 0.3831],
        [0.5841, 0.3831, 1.0000]], device='mps:0')

Other encoder-only models...

following google's footsteps many organisations released their own encoder-only models...

- RoBERTa - facebook AI :
  - similar to bert but better performance across the board (pretrained for longer and on a larger dataset). MLM used for pretraining, but NSP was dropped. Dynamic masking masking tokens on the fly...more randomness in the data, hence more diversity and significantly reducing chances of overfitting
- DistilBERT - HuggingFace
  - Scaled down version of BERT (40% smaller and 60% faster, yet it manages to reach about 97% of BERT's performance on most tasks, making it. a great choice for low-resource environments..)
-  ALBERT by Google research
  - All encoder layers in this model share the same weights making it smaller than bert but not faster. this makes it great for use cases where memory size is limited. Good model for training an encoder-only model from scratch on a GPU with littel VRAM
  - ALBERT also introduced factorized embeddings to reduce the size of the embedding layer: in BERT-large, the vocabulary size is about 30,000 and embedding dim is 1024. ALBERT replaces this with 2 smaller matrices (reducing the embedding size) Albert also replaced NSP with sentence order prediction (SOP) given 2 consecutive sentences: goal is to predict which one comes first.. a much harder task than NSP
- ELECTRA 
  - pretraining technique: replaced token detection (RTD). 
- DeBERTa
  - Fairly large model that beat the sota on many NLU tasks. it removes the usual positional embedding layer, and instead uses relative positional embeddings when computing the attention 

**Decoder-Only Transformers:**
- GPT (Generative Pretraining) - pretrained on a dataset of about 7k books and learned to predict the next token, can be used to generate text one token at a time like the original decoder. Decoder only models - great at text generation tasks such as auto-completion, code generation, question answering and logical reasoning and chatbots.